In [0]:
# ==========================================================================
# PRUEBA AISLADA — Microsoft Graph sobre TU PROPIO tenant (Daily Technology)
# Objetivo: confirmar que /subscribedSkus devuelve prepaidUnits.enabled
#           (contratadas) y consumedUnits (asignadas a usuarios).
# NO requiere GDAP: es tu propio directorio, con consentimiento de admin.
# ==========================================================================
import requests, pandas as pd

# --- credenciales de la app que registraste en TU Entra ID -----------------
# Tenant de Daily Technology (ya prellenado; verifícalo en la app registration)
TENANT_ID     = "7ba64740-cf63-4d5d-9e85-ad0333bdbf67"
CLIENT_ID     = "3cca8622-fb6e-4a0e-bb55-52bc9ddbdaae"
CLIENT_SECRET = ""   # <-- bórralo de la celda al terminar
# --------------------------------------------------------------------------

# 1) Token app-only (client credentials)
tok = requests.post(
    f"https://login.microsoftonline.com/{TENANT_ID}/oauth2/v2.0/token",
    data={"client_id": CLIENT_ID, "client_secret": CLIENT_SECRET,
          "scope": "https://graph.microsoft.com/.default",
          "grant_type": "client_credentials"}, timeout=60)
if tok.status_code != 200:
    raise RuntimeError(f"Token falló ({tok.status_code}): {tok.text[:300]}")
access = tok.json()["access_token"]
print("Token OK")

# 2) subscribedSkus del tenant
r = requests.get("https://graph.microsoft.com/v1.0/subscribedSkus",
                 headers={"Authorization": f"Bearer {access}"}, timeout=60)
if r.status_code != 200:
    # 403 aquí = falta el 'Grant admin consent' de Organization.Read.All
    raise RuntimeError(f"subscribedSkus falló ({r.status_code}): {r.text[:400]}")
data = r.json().get("value", [])
print(f"SKUs en el tenant: {len(data)}")

# 3) Tabla legible: contratadas vs asignadas vs sin asignar
rows = []
for s in data:
    contratadas = (s.get("prepaidUnits") or {}).get("enabled")
    asignadas   = s.get("consumedUnits")
    rows.append({
        "skuPartNumber": s.get("skuPartNumber"),
        "skuId":         s.get("skuId"),
        "contratadas":   contratadas,   # prepaidUnits.enabled
        "asignadas":     asignadas,     # consumedUnits (uso REAL a usuarios)
        "sin_asignar":   (contratadas or 0) - (asignadas or 0),
    })
df = pd.DataFrame(rows).sort_values("sin_asignar", ascending=False)
print("\nSi 'asignadas' es distinto de 'contratadas', Graph SÍ da el uso real:")
display(df)